# Preprocess essential climate variables

This notebook preprocesses data on sea surface salinity, chlorophyll-a concentration and land surface temperature. Before, the Bash and Python scripts `1_download_the_data/download_sea_surface_salinity_data.sh`, `1_download_the_data/download_chlorophyll_a_concentration_data.sh` and `1_download_the_data/download_and_preprocess_land_surface_temperature_data.py` need to be executed to download the respective data.

The Bash scripts `1_download_the_data/download_sea_surface_salinity_data.sh` and `1_download_the_data/download_chlorophyll_a_concentration_data.sh` download all the respective data from 2010 to 2018 at once. `1_download_the_data/download_and_preprocess_land_surface_temperature_data.py` downloads data only for a single month at a time and needs to be executed accordingly to download all the available data from 2010 to 2018 for all months.

## Setup

In [ ]:
# import packages
import gc
import os

import geopandas as gpd
import numpy as np
import pandas as pd
import xarray as xr
from pyproj import CRS

In [ ]:
# make list of desired years
years = list(np.arange(2010, 2019))

In [ ]:
# set min and max values for longitude and latitude
lon_min, lon_max, lat_min, lat_max = 60, 100, 0, 40

In [ ]:
def preprocess_ecv(file, ecv, mask_lon, mask_lat):
    """
    This function preprocesses a given essential climate variable (ecv).
    It first loads a given file into an xarray.
    It then selects the relevant dimensions and ecv and crops the xarray to the relevant bounding box based on given longitude and latitude masks.
    It then extracts year and month from time and drops time.
    Finally, it converts the xarray into a dataframe and drops rows where data on the given ecv is missing.
    """

    all_data = xr.open_dataset(file)
    preprocessed_data = (
        all_data[["time", "lat", "lon", ecv]]
        .where(mask_lon & mask_lat, drop=True)
        .to_dataframe()
        .dropna(subset=[ecv])
        .reset_index()
    )
    preprocessed_data["year"] = preprocessed_data["time"].apply(lambda x: x.year)
    preprocessed_data["month"] = preprocessed_data["time"].apply(lambda x: x.month)
    preprocessed_data = preprocessed_data.drop("time", axis=1)

    return preprocessed_data

## Sea surface salinity

First, a sample is explored before preprocessing all the data and saving it.

In [ ]:
path = "../../data/sea_surface_salinity"

In [ ]:
file = os.path.join(path, str(years[0]), sorted(os.listdir(os.path.join(path, str(years[0]))))[0])

In [ ]:
data = xr.open_dataset(file)

In [ ]:
data

In [ ]:
data.info()

In [ ]:
# create masks to reduce the data to the desired longitudes and latitudes
mask_lon = (data["lon"] >= lon_min) & (data["lon"] <= lon_max)
mask_lat = (data["lat"] >= lat_min) & (data["lat"] <= lat_max)

In [ ]:
%%time

# loop over the desired years, load each file, preprocess it, append it to a geodataframe and save it
for year in years:
    print(f"Processing {year}...")

    files = os.listdir(os.path.join(path, str(year)))
    sss_preprocessed = pd.DataFrame(columns=["lat", "lon", "sss", "year", "month"])

    # load files and apply preprocessing function
    for file in files:
        sss_preprocessed_temp = preprocess_ecv(
            os.path.join(path, str(year), file), "sss", mask_lon, mask_lat
        )
        sss_preprocessed = pd.concat([sss_preprocessed, sss_preprocessed_temp], ignore_index=True)

    # aggregate by year and month
    sss_preprocessed_aggregated = (
        sss_preprocessed.groupby(["lat", "lon", "year", "month"]).aggregate("mean").reset_index()
    )

    # create geodataframe
    sss_preprocessed_aggregated_geo = gpd.GeoDataFrame(
        sss_preprocessed_aggregated,
        geometry=gpd.points_from_xy(
            sss_preprocessed_aggregated.lon, sss_preprocessed_aggregated.lat
        ),
    )
    sss_preprocessed_aggregated_geo.crs = CRS.from_epsg(4326)
    sss_preprocessed_aggregated_geo = sss_preprocessed_aggregated_geo.drop(["lat", "lon"], axis=1)

    # save geodataframe
    sss_preprocessed_aggregated_geo.to_file(os.path.join(path, f"monthly_sss_{year}.shp"))

Check out the last file processed to make sure the output is as desired.

In [ ]:
sss_preprocessed_aggregated_geo.crs

In [ ]:
sss_preprocessed_aggregated_geo.shape

In [ ]:
sss_preprocessed_aggregated_geo.info()

In [ ]:
sss_preprocessed_aggregated_geo.head()

In [ ]:
# free memory
del [
    sss_preprocessed_temp,
    sss_preprocessed,
    sss_preprocessed_aggregated,
    sss_preprocessed_aggregated_geo,
]
gc.collect()

## Chlorophyll-a concentration

First, a sample is explored before preprocessing all the data and saving it.

In [ ]:
path = "../../data/chlorophyll_a_concentration"

In [ ]:
file = os.path.join(path, str(years[0]), sorted(os.listdir(os.path.join(path, str(years[0]))))[0])

In [ ]:
data = xr.open_dataset(file)

In [ ]:
data

In [ ]:
data.info()

In [ ]:
# create masks to reduce the data to the desired longitudes and latitudes
mask_lon = (data["lon"] >= lon_min) & (data["lon"] <= lon_max)
mask_lat = (data["lat"] >= lat_min) & (data["lat"] <= lat_max)

In [ ]:
%%time

# loop over the desired years, load each file, preprocess it, append it to a geodataframe and save it
for year in years:
    print(f"Processing {year}...")

    files = os.listdir(os.path.join(path, str(year)))
    chlora_preprocessed = pd.DataFrame(columns=["lat", "lon", "chlora", "year", "month"])

    # load files and apply preprocessing function
    for file in files:
        chlora_preprocessed_temp = preprocess_ecv(
            os.path.join(path, str(year), file), "chlor_a", mask_lon, mask_lat
        )
        chlora_preprocessed_temp = chlora_preprocessed_temp.rename(columns={"chlor_a": "chlora"})
        chlora_preprocessed = pd.concat(
            [chlora_preprocessed, chlora_preprocessed_temp], ignore_index=True
        )

    # aggregate by year and month
    chlora_preprocessed_aggregated = (
        chlora_preprocessed.groupby(["lat", "lon", "year", "month"]).aggregate("mean").reset_index()
    )

    # compute log
    chlora_preprocessed_aggregated["chlora"] = np.log(chlora_preprocessed_aggregated["chlora"])

    # create geodataframe
    chlora_preprocessed_aggregated_geo = gpd.GeoDataFrame(
        chlora_preprocessed_aggregated,
        geometry=gpd.points_from_xy(
            chlora_preprocessed_aggregated.lon, chlora_preprocessed_aggregated.lat
        ),
    )
    chlora_preprocessed_aggregated_geo.crs = CRS.from_epsg(4326)
    chlora_preprocessed_aggregated_geo = chlora_preprocessed_aggregated_geo.drop(
        ["lat", "lon"], axis=1
    )

    # save geodataframe
    chlora_preprocessed_aggregated_geo.to_file(os.path.join(path, f"monthly_chlora_{year}.shp"))

Check out the last file processed to make sure the output is as desired.

In [ ]:
chlora_preprocessed_aggregated_geo.crs

In [ ]:
chlora_preprocessed_aggregated_geo.shape

In [ ]:
chlora_preprocessed_aggregated_geo.info()

In [ ]:
chlora_preprocessed_aggregated_geo.head()

In [ ]:
# free memory
del [
    chlora_preprocessed_temp,
    chlora_preprocessed,
    chlora_preprocessed_aggregated,
    chlora_preprocessed_aggregated_geo,
]
gc.collect()

## Land surface temperature

First, a sample is explored before preprocessing all the data and saving it.

In [ ]:
path = "../../data/land_surface_temperature"

In [ ]:
file = os.path.join(path, str(years[0]), sorted(os.listdir(os.path.join(path, str(years[0]))))[0])

In [ ]:
data = xr.open_dataset(file)

In [ ]:
data

In [ ]:
data.info()

In [ ]:
%%time

# loop over the desired years, load files, preprocess them, create a geodataframe and save it
for year in years:
    print(f"Processing {year}...")

    # load data
    lst_raw = xr.open_mfdataset(os.path.join(path, str(year)) + "/*.nc")

    # drop na
    lst_preprocessed = lst_raw.to_dataframe().dropna(subset=["lst"]).reset_index()

    # get year and month
    lst_preprocessed["year"] = lst_preprocessed["time"].apply(lambda x: x.year)
    lst_preprocessed["month"] = lst_preprocessed["time"].apply(lambda x: x.month)

    # drop time
    lst_preprocessed = lst_preprocessed.drop("time", axis=1)

    # aggregate by year and month
    lst_preprocessed_aggregated = (
        lst_preprocessed.groupby(["lat", "lon", "year", "month"]).aggregate("mean").reset_index()
    )

    # create geodataframe
    lst_preprocessed_aggregated_geo = gpd.GeoDataFrame(
        lst_preprocessed_aggregated,
        geometry=gpd.points_from_xy(
            lst_preprocessed_aggregated.lon, lst_preprocessed_aggregated.lat
        ),
    )
    lst_preprocessed_aggregated_geo.crs = CRS.from_epsg(4326)
    lst_preprocessed_aggregated_geo = lst_preprocessed_aggregated_geo.drop(["lat", "lon"], axis=1)

    # save geodataframe
    lst_preprocessed_aggregated_geo.to_file(os.path.join(path, f"monthly_lst_{year}.shp"))

Check out the last file processed to make sure the output is as desired.

In [ ]:
lst_preprocessed_aggregated_geo.crs

In [ ]:
lst_preprocessed_aggregated_geo.shape

In [ ]:
lst_preprocessed_aggregated_geo.info()

In [ ]:
lst_preprocessed_aggregated_geo.head()

In [ ]:
# free memory
del [lst_raw, lst_preprocessed, lst_preprocessed_aggregated, lst_preprocessed_aggregated_geo]
gc.collect()